# 🌌 Atmosphäre-Analyse
## Layer 0 – Externe Kosmische Treiber

| Layer | Name | Funktion |
|-------|------|----------|
| **0** | **Externe kosmische Einflüsse** | **Äußerer Modulator – dieser Layer** |
| 1 | Lithosphäre / Geophysik | Grundkörper |
| 2 | Erdoberfläche / Ozeane / Land | Oberfläche |
| 3 | Atmosphäre / Wetter / Gewitter | Wettergeschehen |
| 4 | Ionosphäre | Elektrische Schicht |
| 5 | Global Electric Circuit | Feldkopplung |
| 6 | Resonanz- und Musterfeld | Übergeordnete Muster |
| 7 | Interpretation / Systemzustand | Synthese |

> **Kernfrage Layer 0:** Welche externen Einflüsse verändern die elektromagnetischen und atmosphärischen Zustände der Erde?

In [ ]:
# ============================================================
# SETUP
# ============================================================
import warnings; warnings.filterwarnings('ignore')
import datetime, json, math, requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# --- Projektpfade (CWD-unabhaengig, ohne pip install) ---
import sys, pathlib
_root = pathlib.Path.cwd().resolve()
while not (_root / '.project-root').exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root / 'src'))
from atmosphere.paths import layer_state

print(f'✅ Pakete geladen')
print(f'📅 Analysedatum: {datetime.date.today()}')

---
## 1. Systemstruktur: Die 6 Elemente von Layer 0

In [ ]:
# ============================================================
# SYSTEMDIAGRAMM – FIX: Pfeile als Scatter-Linien statt
# add_annotation(axref='paper') – funktioniert in allen
# Plotly-Versionen ohne ValueError
# ============================================================

elements = [
    {'name': 'Sonnenstrahlung<br>(TSI)',     'x': 0.50, 'y': 0.90, 'color': '#F2A623', 'sz': 60},
    {'name': 'UV / X-Ray',                   'x': 0.18, 'y': 0.74, 'color': '#E85D24', 'sz': 54},
    {'name': 'Sonnenwind<br>(IMF)',           'x': 0.82, 'y': 0.74, 'color': '#F2A623', 'sz': 54},
    {'name': 'Geomag. Stürme<br>(CME, Kp)',  'x': 0.10, 'y': 0.51, 'color': '#7F77DD', 'sz': 56},
    {'name': 'Kosmische<br>Strahlung (GCR)', 'x': 0.90, 'y': 0.51, 'color': '#378ADD', 'sz': 56},
    {'name': 'Planetare Zyklen<br>(Milanković)', 'x': 0.50, 'y': 0.54, 'color': '#639922', 'sz': 54},
]
earth = {'x': 0.50, 'y': 0.16}

fig = go.Figure()

# Verbindungslinien (Pfeile als Linien – kein axref='paper' nötig)
for el in elements:
    dx = earth['x'] - el['x']
    dy = earth['y'] - el['y']
    dist = math.sqrt(dx**2 + dy**2)
    # Linie endet ~0.06 vor dem Erdsystem-Knoten
    t = 0.06 / dist
    ex, ey = earth['x'] - dx * t, earth['y'] - dy * t
    fig.add_trace(go.Scatter(
        x=[el['x'], ex], y=[el['y'], ey], mode='lines',
        line=dict(color=el['color'], width=2), opacity=0.5,
        showlegend=False, hoverinfo='skip'
    ))

# Erdsystem
fig.add_trace(go.Scatter(
    x=[earth['x']], y=[earth['y']], mode='markers+text',
    marker=dict(size=82, color='#1D9E75', opacity=0.90,
                line=dict(color='white', width=2.5)),
    text=['🌍 Erdsystem<br>Layer 1–7'], textposition='middle center',
    textfont=dict(size=10, color='white'), showlegend=False, hoverinfo='skip'
))

# Treiber-Knoten
for el in elements:
    fig.add_trace(go.Scatter(
        x=[el['x']], y=[el['y']], mode='markers+text',
        marker=dict(size=el['sz'], color=el['color'], opacity=0.90,
                    line=dict(color='white', width=2)),
        text=[el['name']], textposition='middle center',
        textfont=dict(size=9.5, color='white'),
        showlegend=False,
        hovertemplate=el['name'].replace('<br>', ' ') + '<extra></extra>'
    ))

# Gruppen-Beschriftungen (nur text-Annotationen, kein axref)
for txt, px, py, col in [
    ('☀️  Elektromagnetische Strahlung', 0.50, 0.99, '#B05010'),
    ('⚡  Partikel & Felder',             0.50, 0.61, '#534AB7'),
    ('🔄  Orbital / Langzeit',            0.50, 0.44, '#3B6D11'),
]:
    fig.add_annotation(x=px, y=py, text=txt, showarrow=False,
                       xref='paper', yref='paper',
                       font=dict(size=11, color=col))

fig.update_layout(
    title=dict(text='Layer 0 – Externe Treiber: Systemstruktur', font=dict(size=16)),
    xaxis=dict(showgrid=False, zeroline=False, visible=False, range=[-0.05, 1.05]),
    yaxis=dict(showgrid=False, zeroline=False, visible=False, range=[0.03, 1.05]),
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    height=530, margin=dict(l=20, r=20, t=55, b=20)
)
fig.show()

---
## 2. Echtdaten abrufen – NOAA SWPC

In [ ]:
# ============================================================
# ECHTDATEN – NOAA SWPC Public APIs (kein API-Key nötig)
# ============================================================

ENDPOINTS = {
    'kp_1m':      'https://services.swpc.noaa.gov/json/planetary_k_index_1m.json',
    'f107_cycle': 'https://services.swpc.noaa.gov/json/solar-cycle/observed-solar-cycle-indices.json',
    'solar_wind': 'https://services.swpc.noaa.gov/json/rtsw/rtsw_wind_1m.json',
    'mag_field':  'https://services.swpc.noaa.gov/json/rtsw/rtsw_mag_1m.json',
    'xray':       'https://services.swpc.noaa.gov/json/goes/primary/xrays-1-day.json',
    'alerts':     'https://services.swpc.noaa.gov/products/alerts.json',
}

def fetch(key, timeout=15):
    url = ENDPOINTS[key]
    try:
        r = requests.get(url, timeout=timeout); r.raise_for_status()
        data = r.json()
        print(f'  ✅ {key:<14} {len(data):>5} Einträge')
        return data
    except Exception as e:
        print(f'  ❌ {key:<14} {e}')
        return None

print('Lade NOAA SWPC Daten...')
raw = {k: fetch(k) for k in ENDPOINTS}
print(f'\n📥 {sum(1 for v in raw.values() if v)}/{len(ENDPOINTS)} Quellen geladen')

In [ ]:
# ============================================================
# DATEN AUFBEREITEN
# ============================================================

def find_time_col(df):
    """Findet die Zeitspalte – sucht nach 'time' im Namen."""
    return next((c for c in df.columns if 'time' in c.lower()), df.columns[0])

# --- Kp ---
# API liefert: time_tag, kp_index, estimated_kp, kp
df_kp = None
if raw['kp_1m']:
    df_kp = pd.DataFrame(raw['kp_1m'])
    print('Kp-Spalten:', df_kp.columns.tolist())
    tc = find_time_col(df_kp)
    kp_candidates = [c for c in df_kp.columns if 'kp' in c.lower() and c != tc]
    kc = 'kp' if 'kp' in df_kp.columns else kp_candidates[-1]
    print(f'  Verwende Spalten: time={tc!r}, kp={kc!r}')
    print(f'  Beispielwerte [{kc}]:', df_kp[kc].head(3).tolist())
    df_kp = df_kp[[tc, kc]].copy()
    df_kp.columns = ['time', 'kp']
    df_kp['time'] = pd.to_datetime(df_kp['time'])
    # NOAA kodiert Kp als '2M', '3+', '1-' → Buchstaben/Zeichen abschneiden
    df_kp['kp'] = df_kp['kp'].astype(str).str.extract(r'([0-9]+(?:\.[0-9]*)?)')[0]
    df_kp['kp'] = pd.to_numeric(df_kp['kp'], errors='coerce')
    df_kp = df_kp.dropna(subset=['kp'])
    # Negative Sentinel-Werte (-1, -999) entfernen
    df_kp = df_kp[df_kp['kp'] >= 0].sort_values('time').tail(1440)
    if df_kp.empty:
        print('  ⚠️  Kp-DataFrame leer nach Bereinigung – versuche kp_index-Spalte')
        # Fallback: kp_index statt kp
        df_kp = pd.DataFrame(raw['kp_1m'])[[tc, 'kp_index']].copy()
        df_kp.columns = ['time', 'kp']
        df_kp['time'] = pd.to_datetime(df_kp['time'])
        df_kp['kp']   = pd.to_numeric(df_kp['kp'], errors='coerce')
        df_kp = df_kp[df_kp['kp'] >= 0].dropna(subset=['kp']).sort_values('time').tail(1440)
    if not df_kp.empty:
        print(f'Kp:    aktuell {df_kp["kp"].iloc[-1]:.2f}, max (24h) {df_kp["kp"].max():.2f}')
    else:
        print('  ❌ Kp-Daten nicht verwendbar')
        df_kp = None

# --- F10.7 ---
# API-Spaltennamen variieren je nach Version; dynamisch erkennen
df_f107 = None
if raw['f107_cycle']:
    df_f107 = pd.DataFrame(raw['f107_cycle'])
    print('F10.7-Spalten:', df_f107.columns.tolist())
    tc = find_time_col(df_f107)
    df_f107['time'] = pd.to_datetime(df_f107[tc])
    # F10.7-Spalte: suche nach 'f10' oder 'flux'
    f107_col = next((c for c in df_f107.columns
                     if ('f10' in c.lower() or 'flux' in c.lower())
                     and 'smooth' not in c.lower()), None)
    if f107_col is None:
        print('  ⚠️  Keine F10.7-Spalte gefunden – überprüfe Spalten oben')
    else:
        df_f107 = df_f107.rename(columns={f107_col: 'f10.7'})
        for col in ['f10.7', 'ssn', 'smoothed_ssn']:
            if col in df_f107.columns:
                df_f107[col] = pd.to_numeric(df_f107[col], errors='coerce')
        # Geglätteter F10.7 – suche Spalte mit 'smooth' + 'f10'
        sm_col = next((c for c in df_f107.columns
                       if 'smooth' in c.lower() and 'f10' in c.lower()), None)
        if sm_col:
            df_f107 = df_f107.rename(columns={sm_col: 'smoothed_f10.7'})
            df_f107['smoothed_f10.7'] = pd.to_numeric(df_f107['smoothed_f10.7'], errors='coerce')
        df_f107 = df_f107.dropna(subset=['f10.7']).sort_values('time').tail(60)
        print(f'F10.7: aktuell {df_f107["f10.7"].iloc[-1]:.1f} sfu')

# --- Solarwind ---
df_sw = None
if raw['solar_wind']:
    df_sw = pd.DataFrame(raw['solar_wind'])
    tc = find_time_col(df_sw)
    df_sw['time'] = pd.to_datetime(df_sw[tc])
    for col in ['speed', 'density', 'temperature']:
        if col in df_sw.columns:
            df_sw[col] = pd.to_numeric(df_sw[col], errors='coerce')
    df_sw = df_sw.sort_values('time').tail(1440)
    if 'speed' in df_sw.columns:
        print(f'SW:    aktuell {df_sw["speed"].dropna().iloc[-1]:.0f} km/s')

# --- IMF / Magnetfeld ---
df_mag = None
if raw['mag_field']:
    df_mag = pd.DataFrame(raw['mag_field'])
    tc = find_time_col(df_mag)
    df_mag['time'] = pd.to_datetime(df_mag[tc])
    for col in ['bt', 'bz_gsm', 'bx_gsm', 'by_gsm']:
        if col in df_mag.columns:
            df_mag[col] = pd.to_numeric(df_mag[col], errors='coerce')
    df_mag = df_mag.sort_values('time').tail(1440)
    if 'bz_gsm' in df_mag.columns:
        bz = df_mag['bz_gsm'].dropna().iloc[-1]
        richtung = 'südwärts ⚠️' if bz < -5 else 'nordwärts ✅' if bz > 0 else 'neutral'
        print(f'Bz:    aktuell {bz:.1f} nT  ({richtung})')

# --- X-Ray ---
df_xray = None
if raw['xray']:
    df_xray = pd.DataFrame(raw['xray'])
    tc = find_time_col(df_xray)
    df_xray['time'] = pd.to_datetime(df_xray[tc])
    fc = next((c for c in df_xray.columns
               if any(k in c.lower() for k in ['flux', 'long', 'energy']) and c != tc),
              df_xray.columns[1])
    df_xray['flux'] = pd.to_numeric(df_xray[fc], errors='coerce')
    df_xray = df_xray.sort_values('time')
    print(f'X-Ray: {len(df_xray)} Werte')

print('\n✅ Aufbereitung abgeschlossen')

---
## 3. Zeitreihen-Dashboard (Echtdaten)

In [ ]:
# ============================================================
# DASHBOARD – Kp / Solarwind / IMF Bz / X-Ray
# ============================================================

panels = [
    (df_kp,   'kp',     'Kp-Index',                      '#7F77DD', 'bar'),
    (df_sw,   'speed',  'Solarwind [km/s]',               '#F2A623', 'line'),
    (df_mag,  'bz_gsm', 'IMF Bz GSM [nT]',               '#378ADD', 'bar'),
    (df_xray, 'flux',   'GOES X-Ray Flux [W/m²] (log)',   '#E85D24', 'line'),
]
active = [(df, col, title, color, kind)
          for df, col, title, color, kind in panels
          if df is not None and col in df.columns]

if not active:
    print('Keine Daten verfügbar.')
else:
    fig = make_subplots(
        rows=len(active), cols=1, shared_xaxes=True,
        subplot_titles=[p[2] for p in active],
        vertical_spacing=0.06
    )
    for i, (df, col, title, color, kind) in enumerate(active, 1):
        ds = df.dropna(subset=[col])
        if kind == 'bar':
            if col == 'kp':
                mc = ['#2ecc71' if v < 4 else '#f39c12' if v < 6 else '#e74c3c'
                      for v in ds[col]]
            elif col == 'bz_gsm':
                mc = ['#e74c3c' if v < -5 else '#2ecc71' if v > 5 else '#95a5a6'
                      for v in ds[col]]
            else:
                mc = color
            fig.add_trace(go.Bar(x=ds['time'], y=ds[col], marker_color=mc,
                                 name=title, opacity=0.75), row=i, col=1)
        else:
            h = color.lstrip('#')
            r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
            fc = f'rgba({r},{g},{b},0.12)'
            fig.add_trace(go.Scatter(x=ds['time'], y=ds[col], name=title,
                                     line=dict(color=color, width=1.5),
                                     fill='tozeroy', fillcolor=fc), row=i, col=1)

        # Schwellenwerte
        if col == 'kp':
            fig.add_hline(y=5, line_dash='dot', line_color='#e74c3c',
                          annotation_text='G1-Sturm', row=i, col=1)
            fig.update_yaxes(range=[0, 9], row=i, col=1)
        elif col == 'speed':
            fig.add_hline(y=500, line_dash='dot', line_color='#E85D24',
                          annotation_text='Erhöhter Wind', row=i, col=1)
        elif col == 'bz_gsm':
            fig.add_hline(y=0, line_color='gray', line_width=0.5, row=i, col=1)
            fig.add_hline(y=-5, line_dash='dot', line_color='#e74c3c',
                          annotation_text='Kopplung', row=i, col=1)
        elif col == 'flux':
            fig.update_yaxes(type='log', row=i, col=1)
            for lvl, lbl, lc in [(1e-4,'X','#c0392b'),(1e-5,'M','#e67e22'),(1e-6,'C','#f1c40f')]:
                fig.add_hline(y=lvl, line_dash='dot', line_color=lc,
                              annotation_text=lbl, row=i, col=1)

    fig.update_layout(
        title=dict(text='Layer 0 – Echtdaten Dashboard (NOAA SWPC, letzte 24h)', font=dict(size=16)),
        height=180 * len(active) + 100,
        showlegend=False,
        plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=70, r=30, t=60, b=40)
    )
    fig.show()

---
## 4. Solarer Zyklus – F10.7 & Sunspot Number

In [ ]:
if df_f107 is not None:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=['F10.7 Radiofluss [sfu]', 'Sunspot Number (SSN)'],
                        vertical_spacing=0.1)

    fig.add_trace(go.Scatter(x=df_f107['time'], y=df_f107['f10.7'],
                             line=dict(color='#F2A623', width=2), name='F10.7',
                             fill='tozeroy', fillcolor='rgba(242,166,35,0.15)'), row=1, col=1)
    if 'smoothed_f10.7' in df_f107.columns:
        fig.add_trace(go.Scatter(x=df_f107['time'], y=df_f107['smoothed_f10.7'],
                                 line=dict(color='#E85D24', width=2.5, dash='dash'),
                                 name='F10.7 geglättet'), row=1, col=1)

    if 'ssn' in df_f107.columns:
        fig.add_trace(go.Bar(x=df_f107['time'], y=df_f107['ssn'],
                             marker_color='#7F77DD', opacity=0.55, name='SSN'), row=2, col=1)
    if 'smoothed_ssn' in df_f107.columns:
        fig.add_trace(go.Scatter(x=df_f107['time'], y=df_f107['smoothed_ssn'],
                                 line=dict(color='#534AB7', width=2.5), name='SSN geglättet'), row=2, col=1)

    fig.add_hline(y=150, line_dash='dot', line_color='#E85D24',
                  annotation_text='Hohe Aktivität', row=1, col=1)
    fig.update_yaxes(title_text='F10.7 [sfu]', row=1, col=1)
    fig.update_yaxes(title_text='SSN', row=2, col=1)
    fig.update_layout(
        title=dict(text='Solarer Zyklus 25 – F10.7 & SSN (Monatsmittel, letzte 60 Monate)', font=dict(size=15)),
        height=480, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=70, r=30, t=60, b=40)
    )
    fig.show()

---
## 5. Aktive NOAA Warnungen

In [ ]:
if raw['alerts']:
    alerts = [a for a in raw['alerts'] if isinstance(a, dict)]
    print(f'📡 NOAA Space Weather Alerts: {len(alerts)} Meldungen\n')
    if alerts:
        for a in alerts[:6]:
            msg = a.get('message', str(a))
            print('─' * 60)
            print(msg[:400])
    else:
        print('✅ Keine aktiven Warnungen.')
else:
    print('Alerts nicht verfügbar.')

---
## 6. Zustandsbewertung & Übergabe an Layer 1

In [ ]:
# ============================================================
# SCHRITT 1 – Rohdaten sammeln mit Quellen-Modus
# Modus: 'primary' | 'fallback_3h' | 'last_valid' | 'missing'
# ============================================================

def safe_last(df, col):
    """Gibt letzten nicht-NaN-Wert einer Spalte zurück, oder None."""
    if df is None or col not in df.columns: return None
    s = df[col].dropna()
    return float(s.iloc[-1]) if not s.empty else None

# Primärwerte
f107_now = safe_last(df_f107, 'f10.7')
bz_now   = safe_last(df_mag,  'bz_gsm')
kp_now   = safe_last(df_kp,   'kp')
sw_now   = safe_last(df_sw,   'speed')

# Quellen-Modi
source_modes = {
    'F10.7':          'primary'  if f107_now is not None else 'missing',
    'Kp':             'primary'  if kp_now   is not None else 'missing',
    'IMF_Bz':         'primary'  if bz_now   is not None else 'missing',
    'SW_speed':       'primary'  if sw_now   is not None else 'missing',
}

# ── Fallback 1: Kp → 3h-Endpunkt ────────────────────────────
if kp_now is None:
    try:
        r = requests.get(
            'https://services.swpc.noaa.gov/json/noaa-planetary-k-index.json',
            timeout=10)
        r.raise_for_status()
        fb = pd.DataFrame(r.json())
        tc = next((c for c in fb.columns if 'time' in c.lower()), fb.columns[0])
        kc = next((c for c in fb.columns if 'kp'   in c.lower() and c != tc), None)
        if kc:
            fb[kc] = fb[kc].astype(str).str.extract(r'([0-9]+(?:\.[0-9]*)?)')[0]
            fb[kc] = pd.to_numeric(fb[kc], errors='coerce')
            fb = fb[fb[kc] >= 0].dropna(subset=[kc])
            if not fb.empty:
                kp_now = float(fb[kc].iloc[-1])
                source_modes['Kp'] = 'fallback_3h'
                print(f'  ↩  Kp Fallback (3h): {kp_now:.1f}')
    except Exception as e:
        print(f'  ⚠️  Kp Fallback fehlgeschlagen: {e}')

# ── Fallback 2: Solarwind → letzter gültiger Wert im DataFrame ─
if sw_now is None and df_sw is not None and 'speed' in df_sw.columns:
    s = df_sw['speed'].dropna()
    if not s.empty:
        sw_now = float(s.iloc[-1])
        source_modes['SW_speed'] = 'last_valid'
        print(f'  ↩  SW Fallback (last_valid): {sw_now:.0f} km/s')

print()
print('Quellen-Modi:', source_modes)

# ============================================================
# SCHRITT 2 – Score nur aus verfügbaren Komponenten
# ============================================================

def norm(v, lo, hi):
    """Normiert auf [0,1]. Gibt None zurück wenn v fehlt."""
    if v is None: return None
    return max(0.0, min(1.0, (v - lo) / (hi - lo)))

# Alle Komponenten mit Rohwert und normiertem Score
COMPONENTS = {
    'Sonnenstrahlung (F10.7)':  (f107_now, norm(f107_now, 60, 250)),
    'Geomagn. Aktivität (Kp)':  (kp_now,   norm(kp_now,   0,   9)),
    'IMF Bz (südwärts)':        (bz_now,   norm(-(bz_now or 0) if bz_now else None, -5, 30)
                                            if bz_now is not None else None),
    'Solarwind Geschwindigkeit': (sw_now,   norm(sw_now, 300, 800)),
}

available   = {k: v[1] for k, v in COMPONENTS.items() if v[1] is not None}
unavailable = [k for k, v in COMPONENTS.items() if v[1] is None]

# Score nur aus verfügbaren Werten (keine Null-Imputation)
layer0_score = round(sum(available.values()) / len(available), 4) if available else None

# Confidence: Anteil verfügbarer Komponenten
confidence   = round(len(available) / len(COMPONENTS), 2)

level = ('unbekannt' if layer0_score is None
         else 'ruhig'   if layer0_score < 0.3
         else 'moderat' if layer0_score < 0.6
         else 'aktiv')

# ============================================================
# SCHRITT 3 – Dominant Driver
# ============================================================

def dominant_driver(kp, f107, bz, sw):
    """Erkennt den stärksten externen Treiber."""
    candidates = []
    if kp   is not None and kp   >= 4:   candidates.append(('geomagnetic',    kp / 9))
    if sw   is not None and sw   >= 500:  candidates.append(('solar_wind',     sw / 800))
    if bz   is not None and bz   <= -5:   candidates.append(('imf_southward',  abs(bz) / 30))
    if f107 is not None and f107 >= 150:  candidates.append(('solar_radiation',f107 / 250))
    return max(candidates, key=lambda x: x[1])[0] if candidates else 'none'

driver = dominant_driver(kp_now, f107_now, bz_now, sw_now)

# ============================================================
# SCHRITT 4 – Downstream-Erwartung
# ============================================================

def downstream_expectation(driver, score, kp, bz, f107):
    exp = {
        'magnetosphere': 'ruhig',
        'ionosphere':    'ruhig',
        'schumann_resonance': 'unverändert',
        'lithosphere_currents': 'niedrig',
    }
    if driver == 'geomagnetic' or (kp and kp >= 5):
        exp['magnetosphere']        = 'komprimiert / Sturm'
        exp['ionosphere']           = 'erhöhte Störung (TEC-Variabilität)'
        exp['schumann_resonance']   = 'erhöhte Amplitude möglich'
        exp['lithosphere_currents'] = 'erhöhte Induktionsströme'
    elif driver == 'imf_southward' or (bz and bz <= -5):
        exp['magnetosphere']        = 'Energie-Einkopplung aktiv'
        exp['ionosphere']           = 'erhöhte Elektrondichte möglich'
        exp['schumann_resonance']   = 'leichte Modulation möglich'
    elif driver == 'solar_radiation' or (f107 and f107 >= 150):
        exp['ionosphere']           = 'erhöhte Ionisierung (F-Schicht)'
        exp['schumann_resonance']   = 'leicht erhöhte Hintergrundaktivität'
    return exp

downstream = downstream_expectation(driver, layer0_score, kp_now, bz_now, f107_now)

# ============================================================
# AUSGABE – Konsole
# ============================================================

W = 58
print('=' * W)
print('LAYER 0 – ZUSTANDSBEWERTUNG')
print('=' * W)
for name, (raw_v, score_v) in COMPONENTS.items():
    if score_v is not None:
        bar = '█' * int(score_v * 20) + '░' * (20 - int(score_v * 20))
        src = source_modes.get(name.split()[0].split('(')[-1].rstrip(')'), 'primary')
        print(f'  {name:<35} {bar}  {score_v:.2f}  [{src}]')
    else:
        print(f'  {name:<35} {"─" * 20}  n/a   [missing]')
print('-' * W)
avail_str = f'{len(available)}/{len(COMPONENTS)} Komponenten'
print(f'  Score (available only):  {layer0_score:.3f}   ({avail_str})')
print(f'  Confidence:              {confidence:.0%}')
print(f'  Level:                   {level.upper()}')
print(f'  Dominant Driver:         {driver}')
print('=' * W)
print('DOWNSTREAM-ERWARTUNG')
print('-' * W)
for k, v in downstream.items():
    print(f'  {k:<28} → {v}')
print('=' * W)

# ============================================================
# RADAR-CHART (nur verfügbare Werte)
# ============================================================

cats = list(available.keys())
vals = list(available.values())
if len(cats) >= 3:
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]], theta=cats + [cats[0]],
        fill='toself', fillcolor='rgba(242,166,35,0.25)',
        line=dict(color='#F2A623', width=2.5), name='Aktuell'
    ))
    fig.add_trace(go.Scatterpolar(
        r=[0.6] * (len(cats) + 1), theta=cats + [cats[0]],
        line=dict(color='#E24B4A', dash='dot', width=1.2),
        mode='lines', name='Aktivitätsschwelle'
    ))
    conf_color = '#2ecc71' if confidence >= 0.75 else '#f39c12' if confidence >= 0.5 else '#e74c3c'
    fig.update_layout(
        title=dict(
            text=(f'Layer 0 – Aktivitätsprofil | Score: {layer0_score:.2f} | '
                  f'{level.upper()} | Confidence: {confidence:.0%} | Driver: {driver}'),
            font=dict(size=13)
        ),
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        height=450, showlegend=True,
        margin=dict(l=60, r=60, t=70, b=40)
    )
    fig.show()
else:
    print('⚠️  Zu wenige Daten für Radar-Chart.')

In [ ]:
# ============================================================
# EXPORT — layer0_state.json
# ============================================================

# ── Kp-Kontext mit mehreren Zeitfenstern ─────────────────────
if df_kp is not None and not df_kp.empty:
    kp_series     = df_kp['kp'].tolist()
    kp_current_1m = round(float(kp_series[-1]),         2)
    kp_max_60min  = round(float(max(kp_series[-60:])),  2) if len(kp_series) >= 2 else kp_current_1m
    kp_max_3h     = round(float(max(kp_series[-180:])), 2) if len(kp_series) >= 2 else kp_current_1m
    kp_mean_3h    = round(float(np.mean(kp_series[-180:])), 2) if len(kp_series) >= 2 else kp_current_1m
else:
    kp_current_1m = kp_now
    kp_max_60min  = kp_now
    kp_max_3h     = kp_now
    kp_mean_3h    = kp_now

kp_context = {
    'Kp_current_1m':     kp_current_1m,
    'Kp_max_60min':      kp_max_60min,
    'Kp_max_3h':         kp_max_3h,
    'Kp_mean_3h':        kp_mean_3h,
    'Kp_used_for_score': round(kp_now, 2) if kp_now is not None else None,
    'Kp_score_method':   source_modes.get('Kp', 'unknown'),
}
print(f'Kp-Kontext:  current={kp_current_1m}  max_60min={kp_max_60min}  max_3h={kp_max_3h}')

# ── State Summary Strings ─────────────────────────────────────
_f107_str = (
    f'Sonnenstrahlung moderat (F10.7={f107_now:.0f} sfu).' if f107_now is not None and 100 <= f107_now < 150 else
    f'Sonnenstrahlung erhöht (F10.7={f107_now:.0f} sfu).'  if f107_now is not None and f107_now >= 150 else
    f'Sonnenstrahlung niedrig (F10.7={f107_now:.0f} sfu).' if f107_now is not None else
    'F10.7 nicht verfügbar.'
)
_kp_str = (
    f'Kp={kp_now:.1f} (ruhig).'         if kp_now is not None and kp_now < 3 else
    f'Kp={kp_now:.1f} (moderat aktiv).'  if kp_now is not None and kp_now < 5 else
    f'Kp={kp_now:.1f} (Sturm).'          if kp_now is not None else
    'Kp nicht verfügbar.'
)
_bz_str = (
    f'IMF Bz={bz_now:.1f} nT – ' + (
        'stark südwärts, geoeffektiv.'         if bz_now <= -10 else
        'südwärts, geoeffektiv.'               if bz_now <= -5  else
        'schwach südwärts, nicht geoeffektiv.' if bz_now <  0   else
        'nordwärts, keine Kopplung.'
    ) if bz_now is not None else 'IMF Bz nicht verfügbar.'
)
_sw_str = (
    f'Solarwind {sw_now:.0f} km/s.' if sw_now is not None else
    'Solarwind-Geschwindigkeit fehlt.'
)

state_summary = ' '.join([
    f'Layer-0-Zustand: {level}.', _f107_str, _kp_str, _bz_str, _sw_str,
    f'Datenvollständigkeit: {confidence:.0%}.'
])

# ── layer0_state ─────────────────────────────────────────────
layer0_state = {
    'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
    'layer': 0,
    'name':  'Externe kosmische Treiber',

    # Score & Qualität
    'score':           layer0_score,
    'level':           level,
    'confidence':      confidence,
    'score_basis':     f'{len(available)}/{len(COMPONENTS)} Komponenten verfügbar',
    'missing_components': unavailable,

    # Komponenten-Scores
    'components': {
        k: round(v, 4) if v is not None else None
        for k, v in {k: COMPONENTS[k][1] for k in COMPONENTS}.items()
    },

    # Rohdaten mit Quellen-Modus
    'raw_values': {
        'F10.7_sfu':    {'value': round(f107_now, 1) if f107_now is not None else None,
                         'source': source_modes['F10.7']},
        'Kp_index':     {'value': round(kp_now,   2) if kp_now   is not None else None,
                         'source': source_modes['Kp']},
        'IMF_Bz_nT':    {'value': round(bz_now,   1) if bz_now   is not None else None,
                         'source': source_modes['IMF_Bz']},
        'SW_speed_kms': {'value': round(sw_now,   0) if sw_now   is not None else None,
                         'source': source_modes['SW_speed']},
    },

    # Kp-Zeitfenster (für Layer 9 Validierung)
    'kp_context': kp_context,

    # Flags
    'flags': {
        'geomagnetic_storm':     (kp_now   >= 5)   if kp_now   is not None else None,
        'solar_flux_high':       (f107_now >= 150)  if f107_now is not None else None,
        'bz_strongly_southward': (bz_now   <= -10) if bz_now   is not None else None,
        'bz_geoeffective':       (bz_now   <= -5)  if bz_now   is not None else None,
        'bz_orientation': (
            'strongly_southward' if bz_now is not None and bz_now <= -10 else
            'southward'          if bz_now is not None and bz_now <= -2  else
            'weak_southward'     if bz_now is not None and bz_now <  0   else
            'northward'          if bz_now is not None and bz_now >= 0   else
            None
        ),
        'fast_solar_wind':       (sw_now   >= 500)  if sw_now   is not None else None,
    },

    # Dominant Driver
    'dominant_driver': driver,

    # Downstream-Erwartung
    'downstream_expectation': downstream,

    # Lesbare Zusammenfassung
    'state_summary': state_summary,
}

with open(layer_state(0), 'w', encoding='utf-8') as f:
    json.dump(layer0_state, f, indent=2, ensure_ascii=False)

print(f'✅ gespeichert: {layer_state(0)}')
print(json.dumps(layer0_state, indent=2, ensure_ascii=False))

---
## Zusammenfassung Layer 0

| Aspekt | Inhalt |
|--------|--------|
| **Datenquellen** | NOAA SWPC (Kp, Solarwind, IMF, X-Ray, SSN, F10.7) |
| **Aktualisierung** | 1-Minuten-Auflösung (Kp, Wind, Mag), Monatsmittel (F10.7) |
| **Ausgabe** | `layer0_state.json` – Score, Rohdaten, Flags |
| **→ Layer 4** | UV/X-Ray → Ionisierungsrate Ionosphäre |
| **→ Layer 5** | Sonnenwind / IMF Bz → Global Electric Circuit |
| **→ Layer 1** | Geomagnetische Induktion → lithosphärische Ströme |

> **Nächster Schritt:** `atmosphere_analysis_layer1.ipynb` – Lithosphäre / Geophysikalischer Grundkörper